# Tabular quality and model pipeline

## Goal

This walkthrough creates a deterministic classification table, introduces realistic quality issues, detects and cleans them, runs lightweight EDA, and builds a leakage-safe scikit-learn pipeline. No files or external services are required.

## Setup

All random operations use a fixed seed so the output is reproducible.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from autoprepml import AutoEDA, AutoFeatureEngine, AutoPrepML, make_model_pipeline

rng = np.random.default_rng(42)
features, target = make_classification(
    n_samples=160, n_features=4, n_informative=3, n_redundant=0,
    n_classes=2, weights=[0.65, 0.35], random_state=42
)
frame = pd.DataFrame(features, columns=["income_score", "tenure_years", "usage_score", "risk_score"])
frame["segment"] = np.where(frame["income_score"] > 0, "standard", "premium")
frame["target"] = target
frame.loc[[3, 17, 88], "income_score"] = np.nan
frame.loc[5, "risk_score"] = 12.0
frame.loc[6, "usage_score"] = -11.0
frame.head()

,income_score,tenure_years,usage_score,risk_score,segment,target
0,-2.299937,1.314392,0.797597,0.289775,premium,0
1,-0.486296,-2.149086,-1.674584,-0.444293,premium,0
2,-0.110191,1.324828,-0.619538,0.500917,premium,0
3,NaN,2.473838,2.063319,-0.259591,standard,1
4,-0.013404,1.126506,-2.681288,-0.603985,premium,0


### Detect and clean

Detection is cached by the data fingerprint. Cleaning imputes missing values and handles categorical columns without modifying the original frame.

In [2]:
preparer = AutoPrepML(frame)
issues = preparer.detect(target_col="target")
cleaned, report = preparer.clean(task="classification", target_col="target")
print({
    "input_shape": frame.shape,
    "missing_columns": list(issues["missing_values"]),
    "outlier_count": issues["outliers"]["outlier_count"],
    "output_shape": cleaned.shape,
    "remaining_missing": int(cleaned.isna().sum().sum()),
    "logged_steps": len(report["logs"]),
})

{'input_shape': (160, 6), 'missing_columns': ['income_score'], 'outlier_count': 8, 'output_shape': (160, 6), 'remaining_missing': 0, 'logged_steps': 5}


### EDA and feature engineering

The EDA result is a JSON-like dictionary that can be persisted by an application. Feature engineering is kept separate from cleaning so the workflow remains explicit.

In [3]:
eda = AutoEDA(cleaned)
eda_result = eda.analyze(
    include_correlations=False, include_distributions=False,
    include_outliers=False, generate_insights=True
)
engine = AutoFeatureEngine(cleaned, target_column="target")
engine.create_interactions(columns=["income_score", "usage_score"], max_interactions=2)
print("EDA shape:", eda_result["basic_stats"]["shape"])
print("insight_count:", len(eda.get_insights()))
print("Engineered shape:", engine.get_features().shape)

EDA shape: (160, 6)
insight_count: 3
Engineered shape: (160, 7)


### Fit a leakage-safe model

The preprocessing transformer is fitted only on the training partition. Unknown categories in future rows are ignored safely.

In [4]:
train, test = train_test_split(cleaned, test_size=0.25, random_state=42, stratify=cleaned["target"])
model_pipeline = make_model_pipeline(
    train, LogisticRegression(max_iter=500, random_state=42), target_col="target"
)
model_pipeline.fit(train.drop(columns="target"), train["target"])
predictions = model_pipeline.predict(test.drop(columns="target"))
print("test_accuracy:", round(accuracy_score(test["target"], predictions), 3))
print("transformed_features:", model_pipeline.named_steps["preprocessor"].get_feature_names_out().shape[0])

test_accuracy: 0.775
transformed_features: 5


## Checks

The cleaned table has no missing values, the target remains available, and the model pipeline can predict held-out rows.

In [5]:
assert cleaned.isna().sum().sum() == 0
assert set(predictions).issubset({0, 1})
print("Tabular workflow checks passed.")

Tabular workflow checks passed.
